# IMUSA V2 — Multi-Account Colab Worker 1 (Folds 0 & 1)
This notebook runs **Stratified 5-Fold Cross-Validation Folds 0 & 1** on Account 1.
Saves fold checkpoints and out-of-fold probability files to `outputs/v2/` and Google Drive.

In [1]:
# 1. Environment & GPU Setup
!nvidia-smi
!pip install -q uv
import os
import sys

if not os.path.exists("imusa-multimodal-sentiment"):
    !git clone https://github.com/shubhojit-mitra-dev/imusa-multimodal-sentiment.git
%cd /content/imusa-multimodal-sentiment
!git pull origin main
!pip install -e libs/imusa
sys.path.insert(0, "/content/imusa-multimodal-sentiment/libs/imusa/src")

Fri Sep 11 18:52:01 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   51C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# 2. Google Drive Integration & Automatic data.zip Handling
import os
import shutil

from google.colab import drive, files

drive.mount("/content/drive", force_remount=False)
gdrive_zip = "/content/drive/MyDrive/data.zip"
local_zip = "/content/imusa-multimodal-sentiment/data.zip"

if os.path.exists(gdrive_zip):
    print("Found data.zip in Google Drive. Copying locally...")
    shutil.copy(gdrive_zip, local_zip)
elif not os.path.exists(local_zip):
    print("data.zip not found in Google Drive (MyDrive/data.zip).")
    print("Please select and upload data.zip from your computer now:")
    uploaded = files.upload()
    for fname in uploaded.keys():
        if fname.endswith(".zip"):
            shutil.move(fname, local_zip)
            break

# Copy to Google Drive for future runs
if os.path.exists(local_zip) and not os.path.exists(gdrive_zip):
    print("Saving data.zip to Google Drive (MyDrive/data.zip) for future runs...")
    try:
        shutil.copy(local_zip, gdrive_zip)
        print("Saved to Google Drive.")
    except Exception as e:
        print(f"Note: Could not copy to Drive: {e}")

# Unzip dataset
!unzip -q -o /content/imusa-multimodal-sentiment/data.zip -d /content/imusa-multimodal-sentiment/
print("Dataset extracted to data/.")

Mounted at /content/drive
Found data.zip in Google Drive. Copying locally...
Dataset extracted to data/.


In [3]:
# 3. Run Fold 0 Training (LP-FT, MuRIL, Label Smoothing, Mixup)
!python scripts/train_kfold.py --fold 0 --model-version v2 --num-folds 5 --epochs 10 --lp-epochs 3

2026-09-11 18:53:23,861 - INFO - --- Starting Stratified K-Fold Training: Fold 1/5 (Version: v2) ---
2026-09-11 18:53:23,862 - INFO - Starting dataset cleaning pipeline on /content/imusa-multimodal-sentiment/data/train/train_punjabi_dataset.csv
2026-09-11 18:53:23,969 - INFO - Saved cleaned dataset (2891 rows) to /content/imusa-multimodal-sentiment/data/processed/train_clean.csv
     IMUSA Dataset Cleaning Report      
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━┓
┃ Pipeline Stage               ┃ Count ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━┩
│ Total Raw Rows Parsed        │  3002 │
│ Dropped (Missing Category)   │     0 │
│ Dropped (Invalid Category)   │     0 │
│ Dropped (Missing Image File) │     0 │
│ Dropped (Duplicates)         │   111 │
│ Final Clean Dataset Size     │  2891 │
└──────────────────────────────┴───────┘
2026-09-11 18:53:24,240 - INFO - HTTP Request: GET https://huggingface.co/api/agent-harnesses "HTTP/1.1 200 OK"
2026-09-11 18:53:24,484 - INFO - HTTP Request: HEAD htt

In [4]:
# 4. Run Fold 1 Training
!python scripts/train_kfold.py --fold 1 --model-version v2 --num-folds 5 --epochs 10 --lp-epochs 3

2026-09-11 19:15:55,792 - INFO - --- Starting Stratified K-Fold Training: Fold 2/5 (Version: v2) ---
2026-09-11 19:15:55,793 - INFO - Starting dataset cleaning pipeline on /content/imusa-multimodal-sentiment/data/train/train_punjabi_dataset.csv
2026-09-11 19:15:55,932 - INFO - Saved cleaned dataset (2891 rows) to /content/imusa-multimodal-sentiment/data/processed/train_clean.csv
     IMUSA Dataset Cleaning Report      
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━┓
┃ Pipeline Stage               ┃ Count ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━┩
│ Total Raw Rows Parsed        │  3002 │
│ Dropped (Missing Category)   │     0 │
│ Dropped (Invalid Category)   │     0 │
│ Dropped (Missing Image File) │     0 │
│ Dropped (Duplicates)         │   111 │
│ Final Clean Dataset Size     │  2891 │
└──────────────────────────────┴───────┘
2026-09-11 19:15:56,233 - INFO - HTTP Request: HEAD https://huggingface.co/google/muril-base-cased/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-09-1

In [5]:
# 5. Normalize outputs, compress Fold 0 & 1 Checkpoints & OOF outputs, and save to Google Drive
import glob
import os
import shutil

os.makedirs("outputs/v2/checkpoints", exist_ok=True)
for f in glob.glob("outputs/v1/checkpoints/best_model_fold_*.pt"):
    dst = os.path.join("outputs/v2/checkpoints", os.path.basename(f))
    if not os.path.exists(dst):
        shutil.copy(f, dst)
for f in glob.glob("outputs/v1/oof_*_fold_*.npy"):
    dst = os.path.join("outputs/v2", os.path.basename(f))
    if not os.path.exists(dst):
        shutil.copy(f, dst)

!zip -r fold_0_1_outputs.zip outputs/v2/checkpoints/best_model_fold_0.pt outputs/v2/checkpoints/best_model_fold_1.pt outputs/v2/oof_probs_fold_*.npy outputs/v2/oof_targets_fold_*.npy
if os.path.exists("/content/drive/MyDrive"):
    shutil.copy("fold_0_1_outputs.zip", "/content/drive/MyDrive/fold_0_1_outputs.zip")
    print("Saved fold_0_1_outputs.zip to Google Drive (MyDrive/fold_0_1_outputs.zip).")

  adding: outputs/v2/checkpoints/best_model_fold_0.pt (deflated 7%)
  adding: outputs/v2/checkpoints/best_model_fold_1.pt (deflated 7%)
  adding: outputs/v2/oof_probs_fold_0.npy (deflated 10%)
  adding: outputs/v2/oof_probs_fold_1.npy (deflated 10%)
  adding: outputs/v2/oof_targets_fold_0.npy (deflated 92%)
  adding: outputs/v2/oof_targets_fold_1.npy (deflated 92%)
Saved fold_0_1_outputs.zip to Google Drive (MyDrive/fold_0_1_outputs.zip).
